# Alpha Engine — P2-E1 Colab Research Exchange Pilot

**Owner-run, private, daily-only.** This notebook is a transport and exact-revision preflight, not a trading service or a public web research endpoint. Only run notebooks/code you trust. No default cloud sync, GPU request, or heavy strategy job.

Before running: freeze an approved package locally using `python -m src.research.cloud_exchange pack`, record its SHA256 from the local output, then copy it manually to your *private* Google Drive Research Exchange folder. Verify licensing and sharing permissions.


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, shutil, subprocess, zipfile

drive.mount('/content/drive')
# Adjust to the REAL path in your mounted MyDrive; do not commit a private folder ID.
EXCHANGE_DIR = Path('/content/drive/MyDrive/CHANGE_ME/Research_Exchange_P2')
INPUT_PACKAGE_NAME = 'CHANGE_ME_APPROVED_INPUT.zip'
EXPECTED_PACKAGE_SHA256 = 'CHANGE_ME_LOCAL_PACK_SHA256'

if 'CHANGE_ME' in str(EXCHANGE_DIR) or 'CHANGE_ME' in INPUT_PACKAGE_NAME or 'CHANGE_ME' in EXPECTED_PACKAGE_SHA256:
    raise RuntimeError('Configure private Drive location, exact package name, and independently recorded SHA256')
drive_package = EXCHANGE_DIR / 'input_packages' / INPUT_PACKAGE_NAME
if not drive_package.is_file():
    raise FileNotFoundError(f'No approved input package at configured location: {drive_package}')
scratch = Path('/content/exchange-input.zip')
if scratch.exists():
    raise FileExistsError('Remove stale scratch file manually before starting another run')
shutil.copyfile(drive_package, scratch)  # Copy once: never train through mounted Drive filesystem
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()
assert sha256(scratch) == EXPECTED_PACKAGE_SHA256, 'Input archive hash differs from independently pinned source'
with zipfile.ZipFile(scratch) as z:
    pinned = json.loads(z.read('manifest.json'))
assert pinned['schema_version'] == 'alpha_research_exchange_v1'
assert pinned['research_only'] is True and pinned['trade_ready'] is False
print('Input archive verified by external package SHA; pinned code:', pinned['repo_revision'])


In [ ]:
checkout = Path('/content/alpha_engine_exchange_checkout')
if checkout.exists():
    raise FileExistsError('Do not reuse a mutable checkout for a new experiment')
subprocess.run(['git', 'clone', '--quiet', '--filter=blob:none', '--no-checkout',
                'https://github.com/liuh886/alpha_engine.git', str(checkout)], check=True)
subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', pinned['repo_revision']], check=True)
actual = subprocess.check_output(['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == pinned['repo_revision'], 'Cloned Git revision mismatch'
subprocess.run(['python', '-m', 'src.research.cloud_exchange', 'verify', '--archive', str(scratch)], cwd=checkout, check=True)
subprocess.run(['python', '-m', 'src.research.cloud_exchange', 'stage', '--archive', str(scratch), '--checkout', str(checkout)], cwd=checkout, check=True)
print('Manifest-backed source files staged into exact checkout; no experiment executed yet.')


## Optional local-equivalent study execution

The bundle is only an input transport. A study must already be implemented and its exact frozen spec present in the checkout (or included and hash-pinned in the approved package). Do not use a fallback data provider, automatically relax model gates, or call untrusted arbitrary commands. The existing Alpha Engine study CLI remains authoritative. If the runtime cannot install pinned dependencies, record `BLOCKED` rather than silently changing the environment.


In [ ]:
RUN_STUDY = False  # Owner must explicitly change after confirming model/data readiness.
STUDY_SPEC_RELATIVE_PATH = 'CHANGE_ME_FROZEN_STUDY_SPEC.json'
if RUN_STUDY:
    spec = checkout / STUDY_SPEC_RELATIVE_PATH
    if not spec.is_file() or not spec.resolve().is_relative_to(checkout.resolve()):
        raise FileNotFoundError('Frozen study spec missing or outside pinned checkout')
    subprocess.run(['python', '-m', 'pip', 'install', '--quiet', 'uv'], check=True)
    subprocess.run(['uv', 'sync', '--frozen'], cwd=checkout, check=True)
    subprocess.run(['uv', 'run', '--frozen', 'alpha', 'research', 'study', 'run',
                    '--spec', str(spec)], cwd=checkout, check=True)
    print('Review the *local* immutable research-study receipt and source/cost gates before any upload.')
else:
    print('Preflight only: no study run, and no data or evidence automatically uploaded to Drive.')


## After the run

Keep the result as *unreviewed incoming evidence*. Before returning to Drive: inspect contents for account/private data, ensure research-only boundaries, copy as a uniquely named artifact (never overwrite) and independently compare its SHA256 after upload. The local Agent performs canonical evidence review and any authorized frontend publication. This notebook intentionally does not perform automatic upload or website updates.
